# Control Test


In [ ]:
# @title Env

!pip install -q "energy-plus-utility @ git+https://github.com/janithcyapa/energy-plus-utility.git@main"
import importlib.metadata
ver = importlib.metadata.version("energy-plus-utility")
print(f"\n✅ Installed 'energy-plus-utility' version: {ver}")

# %%
from eplus import prepare_colab_eplus
prepare_colab_eplus(silent=False)

# %%
# Optional: install control (not used in this notebook but kept for compatibility)
!pip install -q control

# ## 2. Load Model (IDF + Weather)


import types, datetime, requests, io, os, gc
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import traceback
from eplus.core import EPlusUtil

# New

In [44]:
import subprocess
import urllib.request
import os
import shutil

# @title Setup
OUT_DIR = "/simulation/eplus_out"

# =========================================================
# 1. INITIALIZE & CLEAN FIRST (Before downloading files)
# =========================================================
# Initialize the API wrapper and wipe the directory clean
sim = EPlusUtil(verbose=3, out_dir=OUT_DIR)
sim.reset_state()
sim.delete_out_dir()
sim.clear_eplus_outputs(patterns="eplusout.*")

# Recreate the clean directory so we can put files into it
os.makedirs(OUT_DIR, exist_ok=True)

# =========================================================
# 2. FILE PREPARATION
# =========================================================
# --- DYNAMIC PATH RESOLUTION FOR FEDORA ---
home_dir = os.path.expanduser('~')
eplus_install_dir = os.path.join(home_dir, "EnergyPlus-25-1-0")
expand_objects_exe = os.path.join(eplus_install_dir, "ExpandObjects")
local_template_source = os.path.join(eplus_install_dir, "ExampleFiles", "HVACTemplate-5ZonePTAC.idf")

print(f"Using EnergyPlus directory: {eplus_install_dir}")

local_template = os.path.join(OUT_DIR, "in.idf")
local_expanded = os.path.join(OUT_DIR, "expanded.idf")
local_epw = os.path.join(OUT_DIR, "weather.epw")

print(f"Copying local IDF template to {local_template}...")
shutil.copy(local_template_source, local_template)

url_epw = "https://raw.githubusercontent.com/janithcyapa/DHCA-Framework/refs/heads/main/System%20Models/Weather%20Files/LKA_Colombo-Katunayake.434500_SWERA.epw"
print("Downloading weather file...")
urllib.request.urlretrieve(url_epw, local_epw)

# =========================================================
# 3. EXPAND MACROS
# =========================================================
print("Expanding HVACTemplate objects...")

if not os.path.exists(expand_objects_exe):
    raise FileNotFoundError(f"Could not find ExpandObjects at {expand_objects_exe}")

current_dir = os.getcwd()
os.chdir(OUT_DIR) # Move terminal into the directory containing 'in.idf'

try:
    subprocess.run([expand_objects_exe], check=True, capture_output=True)
finally:
    os.chdir(current_dir) # Safely return

if not os.path.exists(local_expanded):
    raise FileNotFoundError("ExpandObjects executed but failed to generate 'expanded.idf'.")
else:
    print("✅ Successfully generated expanded.idf!")

# =========================================================
# 4. LOAD MODEL & SET API SPECS
# =========================================================
# Feed the generated files to the simulator safely
sim.set_model(local_expanded, local_epw)

sim.ensure_output_sqlite()
sim.prepare_run_with_co2(
    outdoor_co2_ppm=420.0,
    wipe_outputs=True,
    activate=True,
    reset=True
)



Initialized StateMixin
Initialized EnergyPlus State.
Initialized IDFMixin
Initialized LoggingMixin
Initialized SimulationMixin
Initialized UtilsMixin
Initialized HandlersMixin
Initialized SQLMixin
Initialized ControlMixin
Initialized OccupancyMixin
Initialized ZoneObserverMixin
EnergyPlus state has been reset.
Deleted output directory: /simulation/eplus_out
Using EnergyPlus directory: /root/EnergyPlus-25-1-0
Copying local IDF template to /simulation/eplus_out/in.idf...
Expanding HVACTemplate objects...
✅ Successfully generated expanded.idf!
EnergyPlus state has been reset.
Model set: IDF='/simulation/eplus_out/expanded.idf', EPW='/simulation/eplus_out/weather.epw', OUT_DIR='/simulation/eplus_out'
EnergyPlus state has been reset.
EnergyPlus state has been reset.


'/simulation/eplus_out/expanded__sqlite__annual_co2_minimal_clean.idf'

In [45]:
# @title Simulation Logger (Updated for PTAC)

# Output Variables for PTACs
specs = [
    # Environment
    {"name": "Site Outdoor Air Relative Humidity", "key": "*"},
    {"name": "Site Outdoor Air Humidity Ratio", "key": "*"},
    {"name": "Schedule Value", "key": "CO2-Outdoor-Actuated"},
    {"name": "Zone Outdoor Air Drybulb Temperature", "key": "*"},


    # Node Physics (The Core Discovery)
    {"name": "System Node Temperature", "key": "*"},
    {"name": "System Node Humidity Ratio", "key": "*"},
    {"name": "System Node CO2 Concentration", "key": "*"},
    {"name": "System Node Mass Flow Rate", "key": "*"},

    # Equipment Performance
    {"name": "Cooling Coil Total Cooling Rate", "key": "*"},
    {"name": "Heating Coil Heating Rate", "key": "*"},
    {"name": "Fan Air Mass Flow Rate", "key": "*"},
    {"name": "Fan Electricity Rate", "key": "*"},

    # Room States
    {"name": "Zone Mean Air Temperature", "key": "*"},
    {"name": "Zone Air Relative Humidity", "key": "*"},
    {"name": "Zone Air CO2 Concentration", "key": "*"},
    {"name": "Zone People Occupant Count", "key": "*"}
]

print("Setup Output Variables")
sim.ensure_output_variables(specs, activate=True)

# Setup Data Storage
sim.sim_log_data = []
sim.current_state = {}

def state_logger(self, state):
    if not self.exchange.api_data_fully_ready(state) or self.exchange.warmup_flag(state):
        return

    # --- ONE-TIME SETUP: DEFINE TARGETS ---
    if not hasattr(self, '_handle_definitions'):
        print("\n[Logger] Registering targets for Zone States and PTAC Metrics...")

        self._handle_definitions = {
            'T_out': ("Zone Outdoor Air Drybulb Temperature", "SPACE1-1"),
            'RH_out_%': ("Site Outdoor Air Relative Humidity", "Environment"),
            'CO2_out': ("Schedule Value", "CO2-Outdoor-Actuated"),
        }

        # Track each zone and its dedicated PTAC unit independently
        for z in ["SPACE1-1", "SPACE2-1", "SPACE3-1", "SPACE4-1", "SPACE5-1"]:
            # --- 1. ZONE STATES ---
            self._handle_definitions[f"{z}_T_in"] = ("Zone Mean Air Temperature", z)
            self._handle_definitions[f"{z}_RH_%"] = ("Zone Air Relative Humidity", z)
            self._handle_definitions[f"{z}_CO2_in"] = ("Zone Air CO2 Concentration", z)
            self._handle_definitions[f"{z}_Occ"] = ("Zone People Occupant Count", z)

            # --- 2. THE MIXER (Pre-Coil) ---
            # Condition of air after mixing Outdoor and Return air
            self._handle_definitions[f"{z}_Mix_T"] = ("System Node Temperature", f"{z} PTAC MIXED AIR OUTLET")
            self._handle_definitions[f"{z}_Mix_W"] = ("System Node Humidity Ratio", f"{z} PTAC MIXED AIR OUTLET")
            self._handle_definitions[f"{z}_Mix_C"] = ("System Node CO2 Concentration", f"{z} PTAC MIXED AIR OUTLET")

            # --- 3. THE COILS (Conditioning) ---
            # Condition of air after being cooled
            self._handle_definitions[f"{z}_Cool_Out_T"] = ("System Node Temperature", f"{z} PTAC COOLING COIL OUTLET")
            self._handle_definitions[f"{z}_Cool_Rate"] = ("Cooling Coil Total Cooling Rate", f"{z} PTAC COOLING COIL")

            # Condition of air after being heated
            self._handle_definitions[f"{z}_Heat_Out_T"] = ("System Node Temperature", f"{z} PTAC HEATING COIL OUTLET")
            self._handle_definitions[f"{z}_Heat_Rate"] = ("Heating Coil Heating Rate", f"{z} PTAC HEATING COIL")

            # --- 4. THE SUPPLY (Into Room) ---
            # Final supply state as it enters the zone node
            self._handle_definitions[f"{z}_T_supply"] = ("System Node Temperature", f"{z} PTAC SUPPLY INLET")
            self._handle_definitions[f"{z}_W_supply"] = ("System Node Humidity Ratio", f"{z} PTAC SUPPLY INLET")
            self._handle_definitions[f"{z}_C_supply"] = ("System Node CO2 Concentration", f"{z} PTAC SUPPLY INLET")
            self._handle_definitions[f"{z}_m_dot"] = ("System Node Mass Flow Rate", f"{z} PTAC SUPPLY INLET")

            # Fan performance
            self._handle_definitions[f"{z}_Fan_Power"] = ("Fan Electricity Rate", f"{z} PTAC SUPPLY FAN")

        self._var_handles = {key: -1 for key in self._handle_definitions.keys()}

    # --- RUNTIME EXTRACTION & DYNAMIC RETRY ---
    day = self.exchange.day_of_year(state)
    time_now = self.exchange.current_time(state)
    hours, mins = divmod(int(time_now * 60), 60)

    row = {
        "timestamp": f"Day {day:03d} {hours:02d}:{mins:02d}",
        "day": day,
        "hour": hours,
        "minute": mins,
        "time_decimal": time_now
    }

    # Fetch handles dynamically
    for key, (type_name, intended_key) in self._handle_definitions.items():
        if self._var_handles[key] <= 0:
            keys_to_test = list(dict.fromkeys([intended_key, intended_key.upper(), intended_key.title(), ""]))
            for k in keys_to_test:
                h = self.exchange.get_variable_handle(state, type_name, k)
                if h > 0:
                    self._var_handles[key] = h
                    break

        handle = self._var_handles[key]
        row[key] = self.exchange.get_variable_value(state, handle) if handle > 0 else np.nan

    self.current_state = row
    self.sim_log_data.append(row)

# Bind and register
import types
sim.state_logger = types.MethodType(state_logger, sim)
sim.register_handlers("after_zone", [{"method_name": "state_logger"}])

Setup Output Variables
EnergyPlus state has been reset.


['state_logger']

In [46]:
# @title Occupancy and CO2 (Unchanged from your code)
import requests
import io
import pandas as pd
import numpy as np
import datetime

def preload_occupancy_csv(sim_obj, url):
    print(f"Downloading CSV from: {url}...")
    try:
        resp = requests.get(url)
        resp.raise_for_status()
        df = pd.read_csv(io.StringIO(resp.text))
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df = df.sort_values('timestamp')
        midnight_start = df['timestamp'].iloc[0].normalize()
        df['rel_seconds'] = (df['timestamp'] - midnight_start).dt.total_seconds()
        sim_obj._occ_duration_sec = 86400.0
        df = df.set_index('rel_seconds')
        sim_obj._preloaded_occ_df = df.drop(columns=['timestamp'])
        print(f"Success! Preloaded {len(df)} rows.")
    except Exception as e:
        print(f"Failed to preload CSV: {e}")

csv_url = "https://raw.githubusercontent.com/janithcyapa/DHCA-Framework/refs/heads/main/System%20Models/Weather%20Files/Occupancy_Dataset.csv"
preload_occupancy_csv(sim, csv_url)

def people_injector(self, state):
    if not self.exchange.api_data_fully_ready(state) or self.exchange.warmup_flag(state):
        return

    if not hasattr(self, '_fast_injector_ready'):
        if not hasattr(self, '_preloaded_occ_df'):
            self._fast_injector_ready = False
            return
        self._zone_occ_rules = {
            "SPACE1-1": {"source": "SPACE1-1", "mult": 1.0,  "min": 0, "max": 5},
            "SPACE2-1": {"source": "SPACE1-1", "mult": 1.5,  "min": 0, "max": 4},
            "SPACE3-1": {"source": "SPACE1-1", "mult": 0.4,  "min": 0, "max": 1},
            "SPACE4-1": {"source": "SPACE1-1", "mult": 1.2,  "min": 0, "max": 3},
            "SPACE5-1": {"source": "SPACE1-1", "mult": 2.0,  "min": 0, "max": 6},
        }
        self._people_handles = {}
        target_zones = list(self._zone_occ_rules.keys())
        try:
            ep_people_names = self.exchange.get_object_names(state, "People") or []
        except Exception:
            ep_people_names = []

        for z in target_zones:
            matched_people = [p for p in ep_people_names if z.replace(" ", "").lower() in p.replace(" ", "").lower()]
            handles = [self.exchange.get_actuator_handle(state, "People", "Number of People", p) for p in matched_people if self.exchange.get_actuator_handle(state, "People", "Number of People", p) != -1]
            if handles:
                self._people_handles[z] = handles

        day = self.exchange.day_of_year(state)
        time_hr = self.exchange.current_time(state)
        self._sim_start_date = datetime.datetime(2002, 1, 1) + datetime.timedelta(days=day - 1, seconds=(int(time_hr * 3600)))
        self._fast_injector_ready = True

    if not self._fast_injector_ready or getattr(self, '_occ_duration_sec', 0) == 0:
        return

    day = self.exchange.day_of_year(state)
    time_hr = self.exchange.current_time(state)
    current_date = datetime.datetime(2002, 1, 1) + datetime.timedelta(days=day - 1, seconds=(int(time_hr * 3600)))
    elapsed_seconds = (current_date - self._sim_start_date).total_seconds()
    loop_sec = elapsed_seconds % self._occ_duration_sec

    df = self._preloaded_occ_df
    valid_indices = df.index[df.index <= loop_sec]
    target_idx = df.index[0] if len(valid_indices) == 0 else valid_indices[-1]
    row = df.loc[target_idx]

    for z, handles in self._people_handles.items():
        rule = self._zone_occ_rules.get(z)
        if rule and rule["source"] in row:
            base_val = float(row[rule["source"]])
            val = 0.0 if base_val == 0 else float(np.clip(np.ceil(base_val * rule["mult"]), rule["min"], rule["max"]))
            for h in handles:
                self.exchange.set_actuator_value(state, h, val / len(handles))

sim.people_injector = types.MethodType(people_injector, sim)
sim.register_handlers("begin", [{"method_name": "people_injector"}])


Success! Preloaded 10129 rows.


['people_injector']

In [47]:
# @title Run Simulation
import pandas as pd
from pathlib import Path

# @title Run Simulation
print("Executing Dry Run...")
# THE FIX: reset=False ensures your Python API callbacks survive the dry run
sim.run_dry_run(include_ems_edd=False, reset=True, design_day=True)
print("sim")
sim.set_simulation_params(
    start=(1, 1),
    end=(1, 7),
    timestep_per_hour = 1,
    start_day_of_week="Sunday",
)

print("Starting EnergyPlus Uncontrolled Simulation...")
res = sim.run_annual()

if res == 0:
    print("Simulation Complete!")

if res == 1:
    err_path = Path(OUT_DIR) / "eplusout.err"
    if err_path.exists():
        print("--- EnergyPlus Error Log ---")
        with open(err_path, 'r') as f:
            print(f.read()[-4000:])
    else:
        print(f"Could not find the error file at: {err_path}")

# Extract Data
if hasattr(sim, 'sim_log_data') and len(sim.sim_log_data) > 0:
    df_log = pd.DataFrame(sim.sim_log_data)
    df_log['Time_Hours'] = (df_log['day'] - df_log['day'].iloc[0]) * 24 + df_log['time_decimal']
    df_log.set_index('Time_Hours', inplace=True)
    df_log.to_csv("MPC_PTAC_Simulation_Log.csv")
    print(f"✅ Data Extracted! Logged {len(df_log)} timesteps.")
    display(df_log)
    df_log.to_csv('log.csv', index=False)
else:
    print("❌ ERROR: Log data is empty! The state_logger callback did not fire.")

Executing Dry Run...
EnergyPlus state has been reset.
sim
EnergyPlus state has been reset.
Starting EnergyPlus Uncontrolled Simulation...
Deleted output file: /simulation/eplus_out/eplusout.err
Deleted output file: /simulation/eplus_out/eplusout.audit
EnergyPlus state has been reset.

[Logger] Registering targets for Zone States and PTAC Metrics...
Simulation Complete!
✅ Data Extracted! Logged 168 timesteps.


,timestamp,day,hour,minute,time_decimal,T_out,RH_out_%,W_out,CO2_out,SPACE1-1_T_in,...,SPACE5-1_Mix_C,SPACE5-1_Cool_Out_T,SPACE5-1_Cool_Rate,SPACE5-1_Heat_Out_T,SPACE5-1_Heat_Rate,SPACE5-1_T_supply,SPACE5-1_W_supply,SPACE5-1_C_supply,SPACE5-1_m_dot,SPACE5-1_Fan_Power
Time_Hours,,,,,,,,,,,,,,,,,,,,,
1.333333,Day 001 01:20,1,1,20,1.333333,24.10195,94.0,NaN,420.0,23.044396,...,420.000000,22.809122,0.0,22.809122,0.0,22.809122,0.017396,420.000000,0.0,0.0
2.500000,Day 001 02:30,1,2,30,2.500000,23.90195,94.0,NaN,420.0,22.974682,...,439.644006,22.842847,0.0,22.842847,0.0,22.842847,0.017458,439.644006,0.0,0.0
3.500000,Day 001 03:30,1,3,30,3.500000,23.80195,95.0,NaN,420.0,22.862714,...,508.316853,22.884277,0.0,22.884277,0.0,22.884277,0.017680,508.316853,0.0,0.0
4.500000,Day 001 04:30,1,4,30,4.500000,23.70195,95.0,NaN,420.0,22.721543,...,581.937935,22.756778,0.0,22.756778,0.0,22.756778,0.017553,581.937935,0.0,0.0
5.500000,Day 001 05:30,1,5,30,5.500000,23.50195,96.0,NaN,420.0,22.601349,...,654.142701,22.655011,0.0,22.655011,0.0,22.655011,0.017460,654.142701,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
164.250000,Day 007 20:15,7,20,15,20.250000,28.00195,74.0,NaN,420.0,26.102951,...,2288.486549,24.590473,0.0,24.590473,0.0,24.590473,0.019687,2288.486549,0.0,0.0
165.333333,Day 007 21:20,7,21,20,21.333333,27.00195,74.0,NaN,420.0,25.328360,...,2324.443203,24.436285,0.0,24.436285,0.0,24.436285,0.019461,2324.443203,0.0,0.0
166.250000,Day 007 22:15,7,22,15,22.250000,26.50195,76.0,NaN,420.0,24.788298,...,2412.296505,24.214768,0.0,24.214768,0.0,24.214768,0.019196,2412.296505,0.0,0.0


In [48]:
# @title 📊 Comprehensive Zone & PTAC Thermodynamic Plotter
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ==========================================
# 🎯 SELECT YOUR TARGET ZONE HERE
# ==========================================
target_zone = "SPACE1-1"
# ==========================================

# Prepare data
df_plot = df_log.sort_index()
time_labels = [f"Day {int(d):02d} {int(h):02d}:{int(m):02d}"
               for d, h, m in zip(df_plot['day'], df_plot['hour'], df_plot['minute'])]

# Create subplots
fig = make_subplots(
    rows=6, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.04,
    specs=[
        [{"secondary_y": False}], # 1. Temperatures
        [{"secondary_y": True}],  # 2. Moisture (W on Left, RH on Right)
        [{"secondary_y": True}],  # 3. IAQ & Occupancy
        [{"secondary_y": False}], # 4. Airflow (m_dot)
        [{"secondary_y": False}], # 5. Thermal Power (Cooling/Heating)
        [{"secondary_y": False}]  # 6. Electrical Power (Fan)
    ],
    subplot_titles=(
        f"1. Temperature Profile ({target_zone})",
        "2. Moisture Dynamics (Humidity Ratio & RH)",
        "3. IAQ & Occupancy (CO2 Concentration)",
        "4. Supply Air Mass Flow Rate",
        "5. PTAC Sensible Thermal Output",
        "6. Component Electrical Power"
    )
)

def add_trace(col_name, name, row, group, color=None, dash=None, sec_y=False, fill=None):
    if col_name in df_plot.columns:
        fig.add_trace(
            go.Scatter(
                x=df_plot.index,
                y=df_plot[col_name],
                name=name,
                legendgroup=group,
                line=dict(color=color, dash=dash, width=2),
                fill=fill
            ),
            row=row, col=1, secondary_y=sec_y
        )

# --- 1. TEMPERATURES (°C) ---
add_trace('T_out', "T_Ambient", 1, "temp", color='white', dash='dash')
add_trace(f'{target_zone}_Mix_T', "T_Mixed (Pre-Coil)", 1, "temp", color='gray')
add_trace(f'{target_zone}_T_supply', "T_Supply (Into Zone)", 1, "temp", color='cyan')
add_trace(f'{target_zone}_T_in', "T_Zone (Room)", 1, "temp", color='#FF5733')

# --- 2. MOISTURE (W and RH) ---
# Humidity Ratio (kgWater/kgDryAir) - The engineering metric for coils
add_trace('W_out', "W_Ambient", 2, "moist", color='white', dash='dash')
add_trace(f'{target_zone}_Mix_W', "W_Mixed", 2, "moist", color='gray')
add_trace(f'{target_zone}_W_supply', "W_Supply", 2, "moist", color='cyan')
# Relative Humidity (%) - The comfort metric
add_trace(f'{target_zone}_RH_%', "RH_Zone", 2, "moist", color='#33FF57', sec_y=True)

# --- 3. IAQ & OCCUPANCY (ppm) ---
add_trace('CO2_out', "CO2_Ambient", 3, "iaq", color='white', dash='dash')
add_trace(f'{target_zone}_Mix_C', "CO2_Mixed", 3, "iaq", color='gray')
add_trace(f'{target_zone}_C_supply', "CO2_Supply", 3, "iaq", color='cyan')
add_trace(f'{target_zone}_CO2_in', "CO2_Zone", 3, "iaq", color='#3357FF')
add_trace(f'{target_zone}_Occ', "Occupancy Count", 3, "iaq", color='yellow', sec_y=True)

# --- 4. AIRFLOW (kg/s) ---
# This is your primary control variable (u) for the MPC
add_trace(f'{target_zone}_m_dot', "Mass Flow Rate", 4, "flow", color='#00f2ff', fill='tozeroy')

# --- 5. THERMAL POWER (W) ---
add_trace(f'{target_zone}_Cool_Rate', "Cooling Rate", 5, "pwr_t", color='blue', fill='tozeroy')
add_trace(f'{target_zone}_Heat_Rate', "Heating Rate", 5, "pwr_t", color='red', fill='tozeroy')

# --- 6. ELECTRICAL POWER (W) ---
add_trace(f'{target_zone}_Fan_Power', "Fan Power", 6, "pwr_e", color='green', fill='tozeroy')

# ==========================================
# 🎨 STYLING & AXIS CONFIG
# ==========================================
fig.update_layout(
    template="plotly_dark", height=1800, hovermode="x unified",
    title=f"Detailed PTAC & Zone Diagnostic: {target_zone}",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

# X-Axis formatting
tick_idx = df_plot.index[::12]
tick_lbl = [time_labels[i] for i in range(0, len(time_labels), 12)]
fig.update_xaxes(showticklabels=True, tickmode='array', tickvals=tick_idx, ticktext=tick_lbl, tickangle=45)

# Y-Axis Titles
fig.update_yaxes(title_text="Temp (°C)", row=1, col=1)
fig.update_yaxes(title_text="Hum. Ratio (kg/kg)", row=2, col=1, secondary_y=False)
fig.update_yaxes(title_text="RH (%)", row=2, col=1, secondary_y=True)
fig.update_yaxes(title_text="CO2 (ppm)", row=3, col=1, secondary_y=False)
fig.update_yaxes(title_text="People", row=3, col=1, secondary_y=True, range=[0, 10])
fig.update_yaxes(title_text="Flow (kg/s)", row=4, col=1)
fig.update_yaxes(title_text="Thermal (Watts)", row=5, col=1)
fig.update_yaxes(title_text="Electric (Watts)", row=6, col=1)

fig.show()

In [ ]:
# @title
import os

expanded_idf_path = "/simulation/eplus_out/expanded.idf"

if os.path.exists(expanded_idf_path):
    print("--- 🕵️‍♂️ RAW SOURCE CODE SCAN FOR SPACE1-1 ---")
    with open(expanded_idf_path, 'r') as f:
        lines = f.readlines()

    for i, line in enumerate(lines):
        if  "HUM" in line.upper():
            print(f"Found Fan Reference: {line.strip()}")
        # Look for the Fan Name
        # if "SPACE1-1" in line.upper() and "FAN" in line.upper():
        #     print(f"Found Fan Reference: {line.strip()}")

        # # Look for the Zone Inlet Node Name
        # if "SPACE1-1" in line.upper() and "INLET" in line.upper():
        #      print(f"Found Inlet Node: {line.strip()}")
else:
    print("Could not find expanded.idf")

# OLD

In [ ]:
# @title
import re
import urllib.request

# 1. Download Files
url_idf = "https://raw.githubusercontent.com/janithcyapa/DHCA-Framework/refs/heads/main/System%20Models/5ZoneAutoDXVAV.idf"
url_epw = "https://raw.githubusercontent.com/janithcyapa/DHCA-Framework/refs/heads/main/System%20Models/Weather%20Files/LKA_Colombo-Katunayake.434500_SWERA.epw"

local_epw = "Colombo_Weather.epw"
local_idf = "MPC_VAV_Dictator.idf"

print("Downloading and processing files...")
urllib.request.urlretrieve(url_epw, local_epw)
idf_text = urllib.request.urlopen(url_idf).read().decode('utf-8')

# 2. Put Thermostats to Sleep (Widen Deadbands to 10C-40C so they never call for air)
idf_text = re.sub(r'21\.1', '10.0', idf_text)
idf_text = re.sub(r'12\.8', '10.0', idf_text)
idf_text = re.sub(r'23\.9', '40.0', idf_text)
idf_text = re.sub(r'32\.2', '40.0', idf_text)

# 3. Hack the VAV Boxes
zones = ["SPACE1-1", "SPACE2-1", "SPACE3-1", "SPACE4-1", "SPACE5-1"]
for z in zones:
    # Highly robust regex to find the specific VAV block regardless of spacing
    pattern = re.compile(
        rf"(AirTerminal:SingleDuct:VAV:Reheat,\s*(?i:{z})\s+VAV Reheat,.*?)"
        rf"(autosize)(\s*!-\s*Maximum Air Flow Rate.*?)"
        rf"(Constant)(\s*!-\s*Zone Minimum Air Flow Input Method.*?)"
        rf"(0\.3)(\s*!-\s*Constant Minimum Air Flow Fraction.*?)"
        rf"(,)(\s*!-\s*Fixed Minimum Air Flow Rate.*?)"
        rf"(,)(\s*!-\s*Minimum Air Flow Fraction Schedule Name)",
        re.DOTALL | re.IGNORECASE
    )

    match = pattern.search(idf_text)
    if match:
        # We lock Maximum Flow to exactly 1.0 m3/s, and set control to Scheduled.
        # This means an API command of 0.15 equals exactly 0.15 m3/s!
        repl = rf"\1 1.0\3Scheduled\5 \7 \9 {z}_MPC_Flow_Cmd\11"
        idf_text = pattern.sub(repl, idf_text, count=1)

        # Inject the command schedule at the end of the file
        idf_text += f"\nSchedule:Constant,\n  {z}_MPC_Flow_Cmd,        !- Name\n  Fraction,                !- Schedule Type Limits Name\n  0.15;                    !- Hourly Value\n"
        print(f"✅ Successfully hacked {z} VAV Box.")
    else:
        print(f"❌ Could not find {z} VAV Box.")

# 4. Save
with open(local_idf, "w") as f:
    f.write(idf_text)
print(f"\n✅ Surgeon Complete. Saved to {local_idf}!")

In [ ]:
# @title Setup
OUT_DIR = "/simulation/eplus_out"
url_idf = "https://raw.githubusercontent.com/janithcyapa/DHCA-Framework/refs/heads/main/System%20Models/5ZoneAutoDXVAV.idf"
url_epw = "https://raw.githubusercontent.com/janithcyapa/DHCA-Framework/refs/heads/main/System%20Models/Weather%20Files/LKA_Colombo-Katunayake.434500_SWERA.epw"

sim = EPlusUtil(verbose=3, out_dir=OUT_DIR)
sim.reset_state()
sim.delete_out_dir()
sim.clear_eplus_outputs(patterns="eplusout.*")
# sim.set_model_from_url(url_idf, url_epw)
sim.set_model(local_idf_path, local_epw_path)

sim.ensure_output_sqlite()
sim.prepare_run_with_co2(
    outdoor_co2_ppm=420.0,
    wipe_outputs=True,
    activate=True,
    reset=True
)

specs = [
    # Environment (Keeping these for W_out and RH_out)
    {"name": "Site Outdoor Air Humidity Ratio", "key": "Environment"},
    {"name": "Site Outdoor Air Relative Humidity", "key": "Environment"},
    {"name": "Schedule Value", "key": "CO2-Outdoor-Actuated"},

    # THE BYPASS: Ask for the weather directly outside the Zone instead of the Site
    {"name": "Zone Outdoor Air Drybulb Temperature", "key": "*"},

    # Nodes & Flow
    {"name": "System Node Temperature", "key": "*"},
    {"name": "System Node Humidity Ratio", "key": "*"},
    {"name": "System Node CO2 Concentration", "key": "*"},
    {"name": "System Node Mass Flow Rate", "key": "*"},
    {"name": "System Node Current Density Volume Flow Rate", "key": "*"},

    # AHU & Zone Energy
    {"name": "Cooling Coil Total Cooling Rate", "key": "*"},
    {"name": "Heating Coil Heating Rate", "key": "*"},
    {"name": "Fan Electricity Rate", "key": "*"},

    # Zone States & Loads
    {"name": "Zone Mean Air Temperature", "key": "*"},
    {"name": "Zone Mean Radiant Temperature", "key": "*"},
    {"name": "Zone Mean Air Humidity Ratio", "key": "*"},
    {"name": "Zone Air Relative Humidity", "key": "*"},
    {"name": "Zone Air CO2 Concentration", "key": "*"},
    {"name": "Zone People Occupant Count", "key": "*"},
    {"name": "Zone Electric Equipment Total Heating Rate", "key": "*"}
]
print("Setup Output Variable")
sim.ensure_output_variables(specs, activate=True)

# Setup Data Storage (Renamed back to sim_log_data)
sim.sim_log_data = []
sim.current_state = {}


In [ ]:
# @title Simulation Logger
def state_logger(self, state):
    if not self.exchange.api_data_fully_ready(state) or self.exchange.warmup_flag(state):
        return

    # --- ONE-TIME SETUP: DEFINE TARGETS ---
    if not hasattr(self, '_handle_definitions'):
        print("\n[Logger] Registering targets for MPC States and AHU Metrics...")

        self._handle_definitions = {
            # THE BYPASS: Map T_out to SPACE1-1's local outdoor temperature
            'T_out': ("Zone Outdoor Air Drybulb Temperature", "SPACE1-1"),

            'W_out': ("Site Outdoor Air Humidity Ratio", "Environment"),
            'RH_out_%': ("Site Outdoor Air Relative Humidity", "Environment"),
            'CO2_out': ("Schedule Value", "CO2-Outdoor-Actuated"),

            'AHU_Mix_Temp': ("System Node Temperature", "Mixed Air Node 1"),
            'AHU_Cool_Out_Temp': ("System Node Temperature", "Main Cooling Coil 1 Outlet Node"),
            'AHU_Heat_Out_Temp': ("System Node Temperature", "Main Heating Coil 1 Outlet Node"),
            'T_s': ("System Node Temperature", "VAV Sys 1 Outlet Node"),
            'W_s': ("System Node Humidity Ratio", "VAV Sys 1 Outlet Node"),
            'C_s': ("System Node CO2 Concentration", "VAV Sys 1 Outlet Node"),

            'AHU_Cooling_Rate': ("Cooling Coil Total Cooling Rate", "Main Cooling Coil 1"),
            'AHU_Heating_Rate': ("Heating Coil Heating Rate", "Main Heating Coil 1"),
            'AHU_Fan_Power': ("Fan Electricity Rate", "Supply Fan 1")
        }

        for z in ["SPACE1-1", "SPACE2-1", "SPACE3-1", "SPACE4-1", "SPACE5-1"]:
            self._handle_definitions[f"{z}_T_in"] = ("Zone Mean Air Temperature", z)
            self._handle_definitions[f"{z}_T_m"] = ("Zone Mean Radiant Temperature", z)
            self._handle_definitions[f"{z}_W_in"] = ("Zone Mean Air Humidity Ratio", z)
            self._handle_definitions[f"{z}_RH_%"] = ("Zone Air Relative Humidity", z)
            self._handle_definitions[f"{z}_CO2_in"] = ("Zone Air CO2 Concentration", z)
            self._handle_definitions[f"{z}_Occ"] = ("Zone People Occupant Count", z)
            self._handle_definitions[f"{z}_Q_equip"] = ("Zone Electric Equipment Total Heating Rate", z)
            self._handle_definitions[f"{z}_V_dot"] = ("System Node Current Density Volume Flow Rate", f"{z} In Node")
            self._handle_definitions[f"{z}_m_dot"] = ("System Node Mass Flow Rate", f"{z} In Node")
            self._handle_definitions[f"{z}_Reheat_Rate"] = ("Heating Coil Heating Rate", f"{z} Zone Coil")

        # Cache array
        self._var_handles = {key: -1 for key in self._handle_definitions.keys()}

    # --- RUNTIME EXTRACTION & DYNAMIC RETRY ---
    day = self.exchange.day_of_year(state)
    time_now = self.exchange.current_time(state)
    hours, mins = divmod(int(time_now * 60), 60)

    row = {
        "timestamp": f"Day {day:03d} {hours:02d}:{mins:02d}",
        "day": day,
        "hour": hours,
        "minute": mins,
        "time_decimal": time_now
    }

    # Fetch handles dynamically
    for key, (type_name, intended_key) in self._handle_definitions.items():
        if self._var_handles[key] <= 0:
            keys_to_test = list(dict.fromkeys([intended_key, intended_key.upper(), intended_key.title(), ""]))
            for k in keys_to_test:
                h = self.exchange.get_variable_handle(state, type_name, k)
                if h > 0:
                    self._var_handles[key] = h
                    break

        handle = self._var_handles[key]
        row[key] = self.exchange.get_variable_value(state, handle) if handle > 0 else np.nan

    self.current_state = row
    self.sim_log_data.append(row)

# Bind and register
sim.state_logger = types.MethodType(state_logger, sim)
sim.register_handlers("after_zone", [{"method_name": "state_logger"}])

print(f"Handlers on 'after_zone' hook: {sim.list_handlers('after_zone')}")

In [ ]:
# @title Occupancy and CO2
def preload_occupancy_csv(sim_obj, url):
    """
    Downloads the CSV, anchors time to midnight, and forces a perfect 24-hour loop.
    """
    print(f"Downloading CSV from: {url}...")
    try:
        resp = requests.get(url)
        resp.raise_for_status()

        df = pd.read_csv(io.StringIO(resp.text))
        if 'timestamp' not in df.columns:
            raise ValueError("CSV must contain a 'timestamp' column.")

        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df = df.sort_values('timestamp')

        # 1. Anchor to MIDNIGHT of the first day to prevent timestep offset
        midnight_start = df['timestamp'].iloc[0].normalize()
        df['rel_seconds'] = (df['timestamp'] - midnight_start).dt.total_seconds()

        # 2. Force exactly 24 hours for daily looping to prevent modulo drift
        sim_obj._occ_duration_sec = 86400.0

        # 3. Clean up dataframe
        df = df.set_index('rel_seconds')
        sim_obj._preloaded_occ_df = df.drop(columns=['timestamp'])

        zones = list(sim_obj._preloaded_occ_df.columns)
        print(f"Success! Preloaded {len(df)} rows. Loop locked to 24.00 hours.")
        print(f"Detected Source Columns: {zones}")

    except Exception as e:
        print(f"Failed to preload CSV: {e}")
# Load CSV
csv_url = "https://raw.githubusercontent.com/janithcyapa/DHCA-Framework/refs/heads/main/System%20Models/Weather%20Files/Occupancy_Dataset.csv"
preload_occupancy_csv(sim, csv_url)

def people_injector(self, state):
    """
    Lightning-fast runtime handler. Maps actuators on the first tick.
    Reads a single baseline zone from the CSV and uses multipliers,
    ceilings, and clipping to populate other zones synthetically.
    """
    if not self.exchange.api_data_fully_ready(state) or self.exchange.warmup_flag(state):
        return

    # --- 1. One-Time Setup: Map Actuators & Define Extrapolation Rules ---
    if not hasattr(self, '_fast_injector_ready'):
        # Ensure the data was preloaded via Step 1
        if not hasattr(self, '_preloaded_occ_df'):
            print("[Injector] ERROR: Data not preloaded. Run preload_occupancy_csv() first.")
            self._fast_injector_ready = False
            return

        # =========================================================
        # CONFIGURATION DICTIONARY: TWEAK YOUR MULTIPLIERS HERE
        # source: The CSV column to read the baseline value from
        # mult: The multiplier applied to the source value
        # min/max: The clipping bounds to enforce physical limits
        # =========================================================
        self._zone_occ_rules = {
            "SPACE1-1": {"source": "SPACE1-1", "mult": 1.0,  "min": 0, "max": 5}, # Baseline
            "SPACE2-1": {"source": "SPACE1-1", "mult": 1.5,  "min": 0, "max": 4},
            "SPACE3-1": {"source": "SPACE1-1", "mult": 0.4,  "min": 0, "max": 1},
            "SPACE4-1": {"source": "SPACE1-1", "mult": 1.2,  "min": 0, "max": 3},
            "SPACE5-1": {"source": "SPACE1-1", "mult": 2.0,  "min": 0, "max": 6},
        }

        self._people_handles = {}
        target_zones = list(self._zone_occ_rules.keys())

        try:
            ep_people_names = self.exchange.get_object_names(state, "People") or []
        except Exception:
            ep_people_names = []

        # Map to EnergyPlus Actuators based on the target zones, NOT just CSV columns
        mapped_count = 0
        for z in target_zones:
            matched_people = [p for p in ep_people_names if z.replace(" ", "").lower() in p.replace(" ", "").lower()]
            handles = []
            for p in matched_people:
                h = self.exchange.get_actuator_handle(state, "People", "Number of People", p)
                if h != -1:
                    handles.append(h)
                    mapped_count += 1
            if handles:
                self._people_handles[z] = handles

        print(f"\n[Injector] Mapped {mapped_count} actuators across {len(self._people_handles)} zones (Extrapolation Active).")

        # Mark exact start time of the simulation
        day = self.exchange.day_of_year(state)
        time_hr = self.exchange.current_time(state)
        self._sim_start_date = datetime.datetime(2002, 1, 1) + datetime.timedelta(days=day - 1, seconds=(int(time_hr * 3600)))

        self._fast_injector_ready = True

    # --- Runtime Safety Check ---
    if not self._fast_injector_ready or getattr(self, '_occ_duration_sec', 0) == 0:
        return

    # --- 2. Calculate Elapsed Time & Loop ---
    day = self.exchange.day_of_year(state)
    time_hr = self.exchange.current_time(state)
    current_date = datetime.datetime(2002, 1, 1) + datetime.timedelta(days=day - 1, seconds=(int(time_hr * 3600)))

    elapsed_seconds = (current_date - self._sim_start_date).total_seconds()
    loop_sec = elapsed_seconds % self._occ_duration_sec

    # --- 3. Fast Data Lookup (Forward Fill) ---
    df = self._preloaded_occ_df
    valid_indices = df.index[df.index <= loop_sec]
    target_idx = df.index[0] if len(valid_indices) == 0 else valid_indices[-1]
    row = df.loc[target_idx]

    # --- 4. Extrapolate and Inject Values ---
    for z, handles in self._people_handles.items():
        rule = self._zone_occ_rules.get(z)
        if not rule:
            continue

        src_col = rule["source"]
        if src_col in row:
            base_val = float(row[src_col])

            # Apply math: Base * Multiplier -> Round Up -> Clip
            if base_val == 0:
                val = 0.0 # Bypasses math to strictly enforce zero at night
            else:
                calculated = np.ceil(base_val * rule["mult"])
                val = float(np.clip(calculated, rule["min"], rule["max"]))

            # Divide evenly if there are multiple People objects in the same room
            per_actuator = val / len(handles)
            for h in handles:
                self.exchange.set_actuator_value(state, h, per_actuator)

# --- Registration ---
sim.people_injector = types.MethodType(people_injector, sim)

sim.register_handlers("begin", [
    {"method_name": "people_injector"},
])

print(f"Handlers on 'begin' hook: {sim.list_handlers("begin")}")

In [ ]:
# @title
import types
def mpc_schedule_controller(self, state):
    if not self.exchange.api_data_fully_ready(state) or self.exchange.warmup_flag(state):
        return

    # --- 1. MAPPING ---
    if not hasattr(self, '_fast_actuators_ready'):
        self._act_handles = {}
        print("\n[Actuator] Mapping VAV Command Schedules...")

        for z in ["SPACE1-1", "SPACE2-1", "SPACE3-1", "SPACE4-1", "SPACE5-1"]:
            sch_name = f"{z}_MPC_Flow_Cmd"
            h = self.exchange.get_actuator_handle(state, "Schedule:Constant", "Schedule Value", sch_name)
            self._act_handles[f"{z}_Flow"] = h
            if h <= 0: print(f"[Actuator] Failed to map: {sch_name}")

        self._fast_actuators_ready = True
        print(f"[Actuator] Mapped Schedules! We are the Dictator.")

    # --- 2. INJECTION ---
    if not hasattr(self, 'mpc_control_actions'):
        # Target flows in Volumetric Rate (m3/s)
        self.mpc_control_actions = {
            "SPACE1-1": 0.15, # 0.15 m3/s
            "SPACE2-1": 0.20, # 0.20 m3/s
            "SPACE3-1": 0.25, # 0.25 m3/s
            "SPACE4-1": 0.10, # 0.10 m3/s
            "SPACE5-1": 0.12  # 0.12 m3/s
        }

    for z in ["SPACE1-1", "SPACE2-1", "SPACE3-1", "SPACE4-1", "SPACE5-1"]:
        handle = self._act_handles.get(f"{z}_Flow", -1)
        if handle > 0:
            target_flow_m3s = self.mpc_control_actions.get(z, 0.15)
            self.exchange.set_actuator_value(state, handle, target_flow_m3s)

# --- Registration ---
sim.mpc_schedule_controller = types.MethodType(mpc_schedule_controller, sim)
sim.register_handlers("before_hvac", [{"method_name": "mpc_schedule_controller"}])

In [ ]:

print("Executing Dry Run...")
sim.run_dry_run(include_ems_edd=False, reset=True, design_day=True)
print("Dry Run Complete!")


# @title Run Simulation
sim.set_simulation_params(
    start=(1, 1),
    end=(1, 7),
    timestep_per_hour = 1, # 4 (every 15 minutes) or 6 (every 10 minutes).
    start_day_of_week="Sunday",
)

print("Starting EnergyPlus Uncontrolled Simulation...")
res = sim.run_annual()

if(res == 0):
    print("Simulation Complete!")

if(res == 1):
    err_path = Path(OUT_DIR) / "eplusout.err"
    if err_path.exists():
        print("--- EnergyPlus Error Log ---")
        with open(err_path, 'r') as f:
            print(f.read()[-4000:])
    else:
        print(f"Could not find the error file at: {err_path}")

In [ ]:
# @title Show Results

# 2. Extract and format the data
if hasattr(sim, 'sim_log_data') and len(sim.sim_log_data) > 0:
    df_log = pd.DataFrame(sim.sim_log_data)

    # Create a continuous time axis for easy plotting
    # using the lowercase 'day' and 'time_decimal' keys from the new logger
    df_log['Time_Hours'] = (df_log['day'] - df_log['day'].iloc[0]) * 24 + df_log['time_decimal']
    df_log.set_index('Time_Hours', inplace=True)

    # 3. Save to CSV
    csv_filename = "MPC_Simulation_Log.csv"
    df_log.to_csv(csv_filename)

    print(f"\n✅ Simulation Complete!")
    print(f"✅ Data Extracted! Shape: {df_log.shape}")
    print(f"✅ Data successfully saved to: {csv_filename}")

    # Display the dataset
    display(df_log)
else:
    print("No log data found. The simulation may not have run correctly.")

In [ ]:
# @title Plot
import plotly.graph_objects as go
from plotly.subplots import make_subplots

selected_zones = [
    "SPACE1-1",
    "SPACE2-1",
    "SPACE3-1",
    "SPACE4-1",
    "SPACE5-1"
    ]

zone_colors = {
    "SPACE1-1": '#FF5733', "SPACE2-1": '#33FF57', "SPACE3-1": '#3357FF',
    "SPACE4-1": '#F033FF', "SPACE5-1": '#33FFF0'
}

# ==========================================
# 📊 PREPARE DATA
# ==========================================
df_plot = df_log.sort_index()

# Simplified time labels: Only HH:MM
time_labels = [f"{int(h):02d}:{int((h%1)*60):02d}" for h in df_plot['time_decimal']]

# ==========================================
# 📈 BUILD SUBPLOTS
# ==========================================
fig = make_subplots(
    rows=6, cols=1,
    shared_xaxes=True,     # Keep zooming synchronized
    vertical_spacing=0.06, # Slightly more space for the extra X-axis labels
    specs=[
        [{"secondary_y": False}], [{"secondary_y": False}],
        [{"secondary_y": True}],  [{"secondary_y": False}],
        [{"secondary_y": False}], [{"secondary_y": False}]
    ],
    subplot_titles=(
        "1. Thermal Dynamics", "2. Moisture Dynamics",
        "3. IAQ & Occupancy", "4. AHU Energy",
        "5. VAV Airflow", "6. Zone Reheat"
    )
)

# Helper function to add traces to groups
def add_trace(trace, row, group):
    trace.legendgroup = group
    # We only want one legend item per group type (e.g. one 'Zone T' in legend)
    # But since you want labels on each, we keep names descriptive
    fig.add_trace(trace, row=row, col=1)

# --- 1. TEMPERATURES ---
add_trace(go.Scatter(x=df_plot.index, y=df_plot['T_out'], name="T_out", line=dict(color='white', dash='dash')), 1, "temp")
add_trace(go.Scatter(x=df_plot.index, y=df_plot['T_s'], name="T_supply", line=dict(color='cyan')), 1, "temp")
for z in selected_zones:
    add_trace(go.Scatter(x=df_plot.index, y=df_plot[f'{z}_T_in'], name=f"{z} T", line=dict(color=zone_colors[z])), 1, "temp")

# --- 2. HUMIDITY ---
add_trace(go.Scatter(x=df_plot.index, y=df_plot['RH_out_%'], name="RH_out", line=dict(color='white', dash='dash')), 2, "humid")
for z in selected_zones:
    add_trace(go.Scatter(x=df_plot.index, y=df_plot[f'{z}_RH_%'], name=f"{z} RH", line=dict(color=zone_colors[z])), 2, "humid")

# --- 3. CO2 & OCCUPANCY ---
add_trace(go.Scatter(x=df_plot.index, y=df_plot['CO2_out'], name="CO2_out", line=dict(color='white', dash='dash')), 3, "iaq")
for z in selected_zones:
    fig.add_trace(go.Scatter(x=df_plot.index, y=df_plot[f'{z}_CO2_in'], name=f"{z} CO2", legendgroup="iaq", line=dict(color=zone_colors[z])), row=3, col=1, secondary_y=False)
    fig.add_trace(go.Scatter(x=df_plot.index, y=df_plot[f'{z}_Occ'], name=f"{z} Occ", legendgroup="iaq", line=dict(color=zone_colors[z], dash='dot'), line_shape='hv'), row=3, col=1, secondary_y=True)

# --- 4. AHU ENERGY ---
add_trace(go.Scatter(x=df_plot.index, y=df_plot['AHU_Cooling_Rate'], name="AHU Cooling", fill='tozeroy', line=dict(color='blue')), 4, "ahu")
add_trace(go.Scatter(x=df_plot.index, y=df_plot['AHU_Heating_Rate'], name="AHU Heating", fill='tozeroy', line=dict(color='red')), 4, "ahu")
add_trace(go.Scatter(x=df_plot.index, y=df_plot['AHU_Fan_Power'], name="AHU Fan", line=dict(color='green')), 4, "ahu")

# --- 5. VAV AIRFLOW ---
for z in selected_zones:
    add_trace(go.Scatter(x=df_plot.index, y=df_plot[f'{z}_m_dot'], name=f"{z} Flow", line=dict(color=zone_colors[z])), 5, "flow")

# --- 6. ZONE REHEAT ---
for z in selected_zones:
    add_trace(go.Scatter(x=df_plot.index, y=df_plot[f'{z}_Reheat_Rate'], name=f"{z} Reheat", fill='tozeroy', line=dict(color=zone_colors[z])), 6, "reheat")

# ==========================================
# 🎨 STYLING & AXIS CONFIG
# ==========================================
fig.update_layout(
    template="plotly_dark",
    height=2000,
    hovermode="x unified",
    legend_tracegroupgap=270, # Offset legends to align with subplots
)

# Show X-axis labels on ALL subplots
fig.update_xaxes(showticklabels=True)

# Formatting X-axis for every plot
fig.update_xaxes(
    tickmode='array',
    tickvals=df_plot.index[::12], # Show label every 3 hours
    ticktext=time_labels[::12],
    tickangle=45,
    title_text="Time (HH:MM)"
)

# Individual Y-axis titles
fig.update_yaxes(title_text="Temp (°C)", row=1, col=1)
fig.update_yaxes(title_text="RH (%)", row=2, col=1)
fig.update_yaxes(title_text="CO2 (ppm)", row=3, col=1, secondary_y=False)
fig.update_yaxes(title_text="People", row=3, col=1, secondary_y=True)
fig.update_yaxes(title_text="Watts", row=4, col=1)
fig.update_yaxes(title_text="kg/s", row=5, col=1)
fig.update_yaxes(title_text="Watts", row=6, col=1)

fig.show()